# AI Daily News — Research Starter

Loads the SQLite archive built nightly by `pipeline.build_db` and
runs a handful of orientation queries: corpus size, daily volume,
top entities, source diversity, and a small tag-cooccurrence graph.

**Prep**: download `archive.db.gz` from the latest GitHub Actions
artifact (see `/research`), gunzip to `archive.db`, then run all cells.

In [ ]:
import sqlite3
import pandas as pd
from pathlib import Path

DB = Path('archive.db')
assert DB.exists(), 'gunzip archive.db.gz into archive.db first'
conn = sqlite3.connect(DB)
pd.read_sql('SELECT * FROM build_meta', conn)

## Daily volume

In [ ]:
vol = pd.read_sql("""
  SELECT day, COUNT(*) AS articles
  FROM articles
  GROUP BY day
  ORDER BY day
""", conn)
vol.plot(x='day', y='articles', kind='bar', figsize=(12,4), title='Articles per day')

## Top entities mentioned

In [ ]:
pd.read_sql("""
  SELECT entity_type, entity, COUNT(*) AS mentions
  FROM entity_mentions
  GROUP BY entity_type, entity
  ORDER BY mentions DESC
  LIMIT 30
""", conn)

## Source diversity

In [ ]:
pd.read_sql("""
  SELECT source_name, COUNT(*) AS articles
  FROM articles
  GROUP BY source_name
  ORDER BY articles DESC
""", conn)

## Category share over time

In [ ]:
cat = pd.read_sql("""
  SELECT day, category, COUNT(*) AS n
  FROM articles
  GROUP BY day, category
  ORDER BY day
""", conn)
cat.pivot(index='day', columns='category', values='n').fillna(0).plot.area(figsize=(12,4), title='Category share')

## Top tag pairs (edges of the tag graph)

In [ ]:
pd.read_sql("""
  SELECT tag_a, tag_b, COUNT(*) AS weight
  FROM tag_cooccurrence
  GROUP BY tag_a, tag_b
  ORDER BY weight DESC
  LIMIT 20
""", conn)

## Corpus completeness — what got dropped

In [ ]:
pd.read_sql("""
  SELECT phase, COUNT(*) AS rows
  FROM skipped
  GROUP BY phase
  ORDER BY rows DESC
""", conn)

In [ ]:
conn.close()